In [4]:
import pandas as pd

In [5]:
# Returns a TextFileReader, which is iterable with chunks of 1000 rows
df = pd.DataFrame()
for chunk in  pd.read_csv('./food_data.csv', on_bad_lines='skip', engine='c', sep='\t', low_memory=False, iterator=True, quotechar='"', chunksize=300000):
    df = pd.concat([df, chunk], ignore_index=True)

In [6]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import MultiLabelBinarizer

class MultiLabelBinarizerTransformer(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.mlb = MultiLabelBinarizer()
    
    def fit(self, X, y=None):
        self.mlb.fit(X)
        return self
    
    def transform(self, X):
        return self.mlb.transform(X)
    
    def get_feature_names_out(self, input_features=None):
        return self.mlb.classes_


In [7]:
from sklearn.preprocessing import StandardScaler, KBinsDiscretizer, FunctionTransformer, OrdinalEncoder
from sklearn.pipeline import Pipeline

# Numerical discretization (e.g., using 10 bins)
discretizer = Pipeline([
    ('discretizer', KBinsDiscretizer(n_bins=10, encode='ordinal', strategy='uniform'))
])

binarizer = Pipeline([
    ('binarizer', MultiLabelBinarizerTransformer())
])

# Categorical preprocessing with One-Hot Encoding
categorical_preprocessor = Pipeline([
    ('ordinal', OrdinalEncoder())
])

In [8]:
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

def prepare_text(text):
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)
    text = ' '.join([word for word in text.split() if word not in ENGLISH_STOP_WORDS])
    return text


text_preprocessor = Pipeline([('lowerizer', FunctionTransformer(lambda x: x.map(prepare_text))),
                              ('tfidf', TfidfVectorizer())])

In [9]:
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression

preprocessor_reg = ColumnTransformer([
    ('text', text_preprocessor, 'product_name'),
    ('discrete_num', discretizer, ['ecoscore_score', 'nutriscore_score', 'nutrition-score-fr_100g']),
    ('text_binarizer', binarizer, 'nutrient_levels_tags'),
    ('cat', categorical_preprocessor, ['nutriscore_grade', 'ecoscore_grade', 'pnns_groups_2', 'pnns_groups_1']) # 
])


preprocessor_cl = ColumnTransformer([
    ('text', text_preprocessor, 'product_name'),
    ('discrete_num', discretizer, ['ecoscore_score', 'nutriscore_score', 'nutrition-score-fr_100g']),
    ('text_binarizer', binarizer, 'nutrient_levels_tags'),
    ('cat', categorical_preprocessor, ['ecoscore_grade', 'pnns_groups_2', 'pnns_groups_1']) # 
])

In [10]:
# Regression model pipeline
regression_pipeline = Pipeline([
    ('preprocess', preprocessor_reg),
    ('regressor', LinearRegression())
])

In [11]:
from sklearn.model_selection import train_test_split

# Split data for regression
X_reg = df[['product_name', 'additives_n', 'nutriscore_score', 'nutriscore_grade',
       'nova_group', 'pnns_groups_1', 'pnns_groups_2', 'ecoscore_score',
       'ecoscore_grade', 'nutrient_levels_tags', 'energy-kcal_100g',
       'fat_100g', 'saturated-fat_100g', 'trans-fat_100g', 'cholesterol_100g',
       'carbohydrates_100g', 'sugars_100g', 'fiber_100g', 'proteins_100g',
       'salt_100g', 'sodium_100g', 'alcohol_100g', 'vitamin-a_100g',
       'vitamin-d_100g', 'vitamin-c_100g', 'potassium_100g', 'calcium_100g',
       'iron_100g', 'fruits-vegetables-nuts-estimate-from-ingredients_100g',
       'nutrition-score-fr_100g']]

y_reg = df['energy-kcal_100g']
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(X_reg, y_reg, test_size=0.2, random_state=42)


In [114]:
# Train the regression model
regression_pipeline.fit(X_train_reg, y_train_reg)

Pipeline(steps=[('preprocess',
                 ColumnTransformer(transformers=[('text',
                                                  Pipeline(steps=[('lowerizer',
                                                                   FunctionTransformer(func=<function <lambda> at 0x00000218A6FB7D90>)),
                                                                  ('tfidf',
                                                                   TfidfVectorizer())]),
                                                  'product_name'),
                                                 ('discrete_num',
                                                  Pipeline(steps=[('discretizer',
                                                                   KBinsDiscretizer(encode='ordinal',
                                                                                    n_bins=10,
                                                                                    strategy='uniform'))]),
                                                  ['ecoscore_score',
                                                   'nutriscore_score',
                                                   'nutrition-score-fr_100g']),
                                                 ('text_binarizer',
                                                  Pipeline(steps=[('binarizer',
                                                                   MultiLabelBinarizerTransformer())]),
                                                  'nutrient_levels_tags'),
                                                 ('cat',
                                                  Pipeline(steps=[('ordinal',
                                                                   OrdinalEncoder())]),
                                                  ['nutriscore_grade',
                                                   'ecoscore_grade',
                                                   'pnns_groups_2',
                                                   'pnns_groups_1'])])),
                ('regressor', LinearRegression())])

In [118]:
# Predict using the regression model
y_pred = regression_pipeline.predict(X_test_reg)

In [126]:
from sklearn.metrics import mean_squared_error
mean_squared_error(y_test_reg, y_pred)

2219680.5536112185

In [26]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.base import clone, BaseEstimator, ClassifierMixin, TransformerMixin
from sklearn.utils.validation import check_X_y, check_is_fitted, check_array
from sklearn.utils.multiclass import check_classification_targets

class KNeighborsOrdinalClassifier(BaseEstimator, ClassifierMixin):
    def __init__(self, n_neighbors=5, *, weights='uniform', 
                 algorithm='auto', leaf_size=30, p=2, 
                 metric='minkowski', metric_params=None, n_jobs=None):
        
        self.n_neighbors = n_neighbors
        self.weights = weights
        self.algorithm = algorithm
        self.leaf_size = leaf_size
        self.p = p
        self.metric = metric
        self.metric_params = metric_params
        self.n_jobs = n_jobs
        
    def fit(self, X, y):
        X, y = check_X_y(X, y)
        check_classification_targets(y)
        
        self.clf_ = KNeighborsClassifier(**self.get_params())
        self.clfs_ = {}
        self.classes_ = np.sort(np.unique(y))
        if self.classes_.shape[0] > 2:
            for i in range(self.classes_.shape[0]-1):
                # for each k - 1 ordinal value we fit a binary classification problem
                binary_y = (y > self.classes_[i]).astype(np.uint8)
                clf = clone(self.clf_)
                clf.fit(X, binary_y)
                self.clfs_[i] = clf
        return self
    
    def predict_proba(self, X):
        X = check_array(X)
        check_is_fitted(self, ['classes_', 'clf_', 'clfs_'])
        
        clfs_predict = {k:self.clfs_[k].predict_proba(X) for k in self.clfs_}
        predicted = []
        for i,y in enumerate(self.classes_):
            if i == 0:
                # V1 = 1 - Pr(y > V1)
                predicted.append(1 - clfs_predict[y][:,1])
            elif y in clfs_predict:
                # Vi = Pr(y > Vi-1) - Pr(y > Vi)
                 predicted.append(clfs_predict[y-1][:,1] - clfs_predict[y][:,1])
            else:
                # Vk = Pr(y > Vk-1)
                predicted.append(clfs_predict[y-1][:,1])
        return np.vstack(predicted).T
    
    def predict(self, X):
        X = check_array(X)
        check_is_fitted(self, ['classes_', 'clf_', 'clfs_'])
        
        return np.argmax(self.predict_proba(X), axis=1)

In [27]:
class DenseTransformer(TransformerMixin):

    def fit(self, X, y=None, **fit_params):
        return self

    def transform(self, X, y=None, **fit_params):
        return X.todense()

In [18]:
from sklearn.model_selection import train_test_split

# Split data for regression
X_cl = df[['product_name', 'additives_n', 'nutriscore_score', 'ecoscore_grade',
       'nova_group', 'pnns_groups_1', 'pnns_groups_2', 'ecoscore_score',
        'nutrient_levels_tags', 'energy-kcal_100g',
       'fat_100g', 'saturated-fat_100g', 'trans-fat_100g', 'cholesterol_100g',
       'carbohydrates_100g', 'sugars_100g', 'fiber_100g', 'proteins_100g',
       'salt_100g', 'sodium_100g', 'alcohol_100g', 'vitamin-a_100g',
       'vitamin-d_100g', 'vitamin-c_100g', 'potassium_100g', 'calcium_100g',
       'iron_100g', 'fruits-vegetables-nuts-estimate-from-ingredients_100g', 'energy-kcal_100g',
       'nutrition-score-fr_100g']]

y_cl = df['nutriscore_grade']
X_train_cl, X_test_cl, y_train_cl, y_test_cl = train_test_split(X_cl, y_cl, test_size=0.2, random_state=42)


In [32]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC

# Classification model pipeline (choose appropriate model based on scaling requirement)
classification_pipeline = Pipeline([
    ('preprocess', preprocessor_cl),
    # ('to_dense', DenseTransformer()), 
    ('classifier', LinearSVC())  # Or KNeighborsClassifier() if scaling ordinal data
])

In [33]:
classification_pipeline.fit(X_train_cl, y_train_cl)

Pipeline(steps=[('preprocess',
                 ColumnTransformer(transformers=[('text',
                                                  Pipeline(steps=[('lowerizer',
                                                                   FunctionTransformer(func=<function <lambda> at 0x000002227706B2E0>)),
                                                                  ('tfidf',
                                                                   TfidfVectorizer())]),
                                                  'product_name'),
                                                 ('discrete_num',
                                                  Pipeline(steps=[('discretizer',
                                                                   KBinsDiscretizer(encode='ordinal',
                                                                                    n_bins=10,
                                                                                    strategy='uniform'))]),
                                                  ['ecoscore_score',
                                                   'nutriscore_score',
                                                   'nutrition-score-fr_100g']),
                                                 ('text_binarizer',
                                                  Pipeline(steps=[('binarizer',
                                                                   MultiLabelBinarizerTransformer())]),
                                                  'nutrient_levels_tags'),
                                                 ('cat',
                                                  Pipeline(steps=[('ordinal',
                                                                   OrdinalEncoder())]),
                                                  ['ecoscore_grade',
                                                   'pnns_groups_2',
                                                   'pnns_groups_1'])])),
                ('classifier', LinearSVC())])

In [36]:
y_cl_pred = classification_pipeline.predict(X_test_cl)

In [38]:
from sklearn.metrics import accuracy_score

accuracy_score(y_test_cl, y_cl_pred)

0.7436510768033212